In [ ]:
# Komórka 1 - Instalacja bibliotek
# !pip install pandas numpy matplotlib seaborn pyxdf scipy

In [ ]:
# KOMÓRKA 2 - Importowanie bibliotek
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyxdf import load_xdf
import json
import os
from scipy.ndimage import gaussian_filter
import warnings
warnings.filterwarnings('ignore')

print("✅ Wszystkie biblioteki załadowane!")

In [ ]:
# KOMÓRKA 3 - Definiowanie ścieżek do plików
PARTICIPANT_ID = ""
BASE_PATH = "../data/"
STIMULI_PATH = "../stimuli/"

TOBII_FILE = f"{BASE_PATH}{PARTICIPANT_ID}.csv"
XDF_FILE = f"{BASE_PATH}{PARTICIPANT_ID}.xdf"
META_FILE = f"{BASE_PATH}{PARTICIPANT_ID}-Procedure_info.csv"

print("📁 Sprawdzanie plików...")
for file_path in [TOBII_FILE, XDF_FILE, META_FILE, STIMULI_PATH]:
    if os.path.exists(file_path):
        print(f"✅ Znaleziono: {file_path}")
    else:
        print(f"❌ Brak pliku: {file_path}")

OUTPUT_DIR = "./results/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# KOMÓRKA 4 - Wczytanie metadanych
meta = pd.read_csv(META_FILE, sep=";")
print("📊 Metadane uczestnika:")
print(meta)

ID = meta.loc[0, "ID"]
AGE = meta.loc[0, "AGE"]
SEX = meta.loc[0, "SEX"]
START_TIME = meta.loc[0, "TIMESTAMP"]

print(f"\n👤 Uczestnik: {ID}, Wiek: {AGE}, Płeć: {SEX}")

In [ ]:
# KOMÓRKA 5 - Wczytanie danych z Tobii
df_tobii = pd.read_csv(TOBII_FILE, header=0, skipinitialspace=True, engine="python")
print(f"📈 Wczytano {len(df_tobii)} wierszy")
print("🔍 Podgląd pierwszych 5 wierszy:")
print(df_tobii.head())

In [ ]:
# KOMÓRKA 6 - Przetwarzanie danych z Tobii
# Konwersja czasów
df_tobii["device_time_stamp"] = pd.to_numeric(df_tobii["device_time_stamp"], errors="coerce")
df_tobii["system_time_stamp"] = pd.to_numeric(df_tobii["system_time_stamp"], errors="coerce")

t0 = df_tobii["device_time_stamp"].iloc[0]
df_tobii["t_seconds_tobii"] = (df_tobii["device_time_stamp"] - t0) / 1e6
print(f"⏱️  Czas rozpoczęcia nagrywania: {t0}")
print(f"📊 Zakres czasowy: {df_tobii['t_seconds_tobii'].min():.1f}s - {df_tobii['t_seconds_tobii'].max():.1f}s")

# Funkcja do bezpiecznego parsowania kolumn z tuplami
def parse_gaze_column(series):
    split = series.str.replace("(", "", regex=False).str.replace(")", "", regex=False).str.split(",", expand=True)
    x = pd.to_numeric(split[0], errors='coerce')
    y = pd.to_numeric(split[1], errors='coerce')
    return x, y

# Parsowanie lewego i prawego oka
if 'left_gaze_point_on_display_area' in df_tobii.columns:
    lx, ly = parse_gaze_column(df_tobii['left_gaze_point_on_display_area'])
    df_tobii['lx'] = lx
    df_tobii['ly'] = ly
    print("✅ Lewe oko – sparsowano")
if 'right_gaze_point_on_display_area' in df_tobii.columns:
    rx, ry = parse_gaze_column(df_tobii['right_gaze_point_on_display_area'])
    df_tobii['rx'] = rx
    df_tobii['ry'] = ry
    print("✅ Prawe oko – sparsowano")

# Filtracja na podstawie validity
if 'left_gaze_point_validity' in df_tobii.columns and 'right_gaze_point_validity' in df_tobii.columns:
    df_tobii = df_tobii[
        (df_tobii['left_gaze_point_validity'] == 1) | 
        (df_tobii['right_gaze_point_validity'] == 1)
    ].copy()
    print(f"✅ Po filtracji validity: {len(df_tobii)} wierszy")

# Uśrednienie pozycji z obu oczu
if 'lx' in df_tobii.columns and 'rx' in df_tobii.columns:
    df_tobii['gaze_x'] = df_tobii[['lx', 'rx']].mean(axis=1, skipna=True)
    df_tobii['gaze_y'] = df_tobii[['ly', 'ry']].mean(axis=1, skipna=True)
elif 'lx' in df_tobii.columns:
    df_tobii['gaze_x'] = df_tobii['lx']
    df_tobii['gaze_y'] = df_tobii['ly']
elif 'rx' in df_tobii.columns:
    df_tobii['gaze_x'] = df_tobii['rx']
    df_tobii['gaze_y'] = df_tobii['ry']
else:
    print("⚠️  Brak danych do uśrednienia")

valid_coords = df_tobii[['gaze_x', 'gaze_y']].notna().all(axis=1).sum()
print(f"✅ Poprawnych współrzędnych: {valid_coords}/{len(df_tobii)} ({valid_coords/len(df_tobii)*100:.1f}%)")

In [ ]:
# Komórka 7 - Wczytanie danych z XDF
print(f"📂 Wczytywanie pliku XDF: {XDF_FILE}")
streams, header = load_xdf(XDF_FILE)

print("\n🔍 Znalezione strumienie:")
for i, stream in enumerate(streams):
    name = stream["info"]["name"][0] if stream["info"]["name"] else f"Strumień_{i}"
    print(f"  {i}: {name}")

# Znajdź strumienie Events i Procedure
events_stream = None
procedure_stream = None
for stream in streams:
    name = stream["info"]["name"][0] if stream["info"]["name"] else ""
    if 'Events' in name:
        events_stream = stream
    elif 'Procedure' in name:
        procedure_stream = stream

# Przetwarzanie Events (zostawiamy jako pomocnicze, ale nie używamy do bodźców)
events_data = []
if events_stream:
    for time, marker in zip(events_stream['time_stamps'], events_stream['time_series']):
        marker_value = marker[0] if isinstance(marker, (list, np.ndarray)) and len(marker) > 0 else marker
        events_data.append({'timestamp': time, 'event': str(marker_value)})
    df_events = pd.DataFrame(events_data)
    print(f"✅ Events: {len(df_events)} zdarzeń")
else:
    df_events = pd.DataFrame()

# Przetwarzanie Procedure – parsowanie JSON
procedure_data = []
if procedure_stream:
    for time, marker in zip(procedure_stream['time_stamps'], procedure_stream['time_series']):
        if isinstance(marker, (list, np.ndarray)) and len(marker) > 0:
            marker_str = marker[0]
        else:
            marker_str = marker
        try:
            data = json.loads(marker_str)
            data['timestamp'] = time  # dodajemy oryginalny czas
            procedure_data.append(data)
        except:
            # Jeśli nie JSON, zapisujemy surowy tekst
            procedure_data.append({'timestamp': time, 'raw': marker_str})
    df_procedure = pd.DataFrame(procedure_data)
    print(f"✅ Procedure: {len(df_procedure)} wpisów")
    print("Przykładowe wpisy:")
    print(df_procedure.head())
else:
    df_procedure = pd.DataFrame()
    print("⚠️  Nie znaleziono strumienia Procedure")

In [ ]:
# KOMÓRKA 8 - Synchronizacja czasów (lepsza metoda)

print("🔄 Synchronizacja czasów...")

if len(df_procedure) > 0:
    
    # pierwszy bodziec w procedurze
    first_stim_time = df_procedure['timestamp'].iloc[0]

    # pierwszy timestamp tobii
    first_tobii_time = df_tobii['device_time_stamp'].iloc[0]

    # przeliczenie Tobii na sekundy
    tobii_start_seconds = first_tobii_time / 1e6

    time_offset = first_stim_time - tobii_start_seconds

    print(f"⏱️ Offset czasu: {time_offset:.3f}s")

    df_procedure['t_sync'] = df_procedure['timestamp'] - first_stim_time

    df_tobii['t_sync'] = df_tobii['t_seconds_tobii']

    print("✅ Synchronizacja zakończona")

else:
    print("⚠️ Brak danych Procedure — brak synchronizacji")

In [ ]:
# Komórka 9 - Łączenie danych i dodawanie metadanych
# Łączymy dane Tobii z Events (jeśli chcemy mieć informacje o ogólnych etapach)
df_combined = df_tobii.copy()

if len(df_events) > 0 and 't_sync' in df_events.columns:
    df_combined = pd.merge_asof(
        df_combined.sort_values('t_seconds_tobii'),
        df_events.sort_values('t_sync')[['t_sync', 'event']],
        left_on='t_seconds_tobii',
        right_on='t_sync',
        direction='nearest',
        tolerance=0.1
    ).rename(columns={'event': 'current_event'})
else:
    df_combined['current_event'] = None

df_combined["participant_id"] = ID
df_combined["age"] = AGE
df_combined["sex"] = SEX
df_combined["study_start_time"] = START_TIME

In [ ]:
# Komórka 10 - Zapis do pliku CSV
output_file = f"{OUTPUT_DIR}combined_data_{ID}.csv"
df_combined.to_csv(output_file, index=False)
print(f"💾 Zapisano do: {output_file}")

In [ ]:
# Komórka 11 - Przygotowanie danych do heatmapy
df_analysis = df_combined.copy()
initial_count = len(df_analysis)
df_analysis = df_analysis.dropna(subset=['gaze_x', 'gaze_y'])
print(f"📉 Usunięto {initial_count - len(df_analysis)} wierszy bez współrzędnych")

x_min, x_max = df_analysis['gaze_x'].min(), df_analysis['gaze_x'].max()
y_min, y_max = df_analysis['gaze_y'].min(), df_analysis['gaze_y'].max()
print(f"📍 Zakres X: {x_min:.3f} do {x_max:.3f}")
print(f"📍 Zakres Y: {y_min:.3f} do {y_max:.3f}")

# Flaga: czy poza ekranem
df_analysis['outside_screen'] = (
    (df_analysis['gaze_x'] < 0) | (df_analysis['gaze_x'] > 1) |
    (df_analysis['gaze_y'] < 0) | (df_analysis['gaze_y'] > 1)
).astype(int)
outside_pct = df_analysis['outside_screen'].mean() * 100
print(f"👀 Procent czasu poza ekranem: {outside_pct:.2f}%")

# Usuwamy punkty poza ekranem
before = len(df_analysis)

df_analysis = df_analysis[
    (df_analysis['gaze_x'] >= 0) &
    (df_analysis['gaze_x'] <= 1) &
    (df_analysis['gaze_y'] >= 0) &
    (df_analysis['gaze_y'] <= 1)
]

after = len(df_analysis)

print(f"🧹 Usunięto {before-after} punktów poza ekranem")

# Kolumny znormalizowane (to samo co gaze_x, gaze_y – tylko dla czytelności)
df_analysis['gaze_x_norm'] = df_analysis['gaze_x']
df_analysis['gaze_y_norm'] = df_analysis['gaze_y']

print(f"✅ Dane gotowe: {len(df_analysis)} wierszy")

In [ ]:
# Komórka 12 - Wyodrębnianie bodźców z Procedure
if len(df_procedure) > 0 and 'STIM-TYPE' in df_procedure.columns:
    stimuli_df = df_procedure[df_procedure['STIM-TYPE'].notna()].copy()
    stimuli_df = stimuli_df.sort_values('t_sync').reset_index(drop=True)
    print(f"🖼️  Znaleziono {len(stimuli_df)} bodźców w Procedure")
    
    stimuli_list = []
    for i in range(len(stimuli_df)):
        stim_name = stimuli_df.loc[i, 'STIM-TYPE']
        start_time = stimuli_df.loc[i, 't_sync']
        
        # Koniec bodźca = początek następnego (lub koniec nagrania)
        if i < len(stimuli_df) - 1:
            end_time = stimuli_df.loc[i+1, 't_sync']
        else:
            end_time = df_tobii['t_seconds_tobii'].max()
        
        stimuli_list.append({
            'stimulus': stim_name,
            'start': start_time,
            'end': end_time,
            'set': stimuli_df.loc[i, 'STIM-SET'] if 'STIM-SET' in stimuli_df.columns else None,
            'cond': stimuli_df.loc[i, 'STIM-COND'] if 'STIM-COND' in stimuli_df.columns else None
        })
    
    print("Przykładowe bodźce:")
    for s in stimuli_list[:5]:
        print(f"  {s['stimulus']}: {s['start']:.2f}s - {s['end']:.2f}s")
else:
    stimuli_list = []
    print("⚠️  Brak danych o bodźcach w Procedure")

In [ ]:
# Komórka 13 - Funkcja do tworzenia heatmapy
def create_simple_heatmap(data, title="Heatmapa", save_path=None):
    if len(data) < 10:
        print(f"⚠️  Za mało danych: {len(data)} punktów")
        return None
    
    x = data['gaze_x_norm'].values
    y = data['gaze_y_norm'].values
    
    heatmap, xedges, yedges = np.histogram2d(
        x, y, bins=150, range=[[0, 1], [0, 1]]
    )
    heatmap = heatmap.T
    heatmap = gaussian_filter(heatmap, sigma=1.5)
    
    plt.figure(figsize=(10, 8))
    plt.imshow(heatmap, origin='lower', extent=[0,1,0,1], cmap='hot', aspect='auto')
    plt.scatter(x, y, s=1, c='blue', alpha=0.1, marker='.')
    plt.colorbar(label='Częstotliwość fiksacji')
    plt.title(title, fontsize=14, pad=20)
    plt.xlabel('Pozycja X (0-1)')
    plt.ylabel('Pozycja Y (0-1)')
    plt.grid(True, alpha=0.3, linestyle='--')
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"💾 Zapisano: {save_path}")
    
    plt.show()
    return heatmap

In [ ]:
# Komórka 14 - Heatmapa na obrazie bodźca (np. twarzy)

from PIL import Image

def create_face_heatmap(data, image_path, title="", save_path=None):

    if len(data) < 10:
        print(f"⚠️ Za mało danych: {len(data)} punktów")
        return None

    img = Image.open(image_path)
    img = np.array(img)

    h, w = img.shape[:2]

    x = data['gaze_x_norm'].values * w
    y = data['gaze_y_norm'].values * h

    heatmap, xedges, yedges = np.histogram2d(
        x, y,
        bins=[w//10, h//10],
        range=[[0, w], [0, h]]
    )

    heatmap = gaussian_filter(heatmap.T, sigma=6)

    plt.figure(figsize=(8,8))

    plt.imshow(img)
    plt.imshow(
        heatmap,
        cmap="jet",
        alpha=0.5,
        extent=[0, w, h, 0]
    )

    plt.title(title)
    plt.axis("off")

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"💾 Zapisano: {save_path}")

    plt.show()

    return heatmap

In [ ]:
def create_stimulus_heatmaps(df_analysis, stimuli_list, output_dir, participant_id):

    for i, stim in enumerate(stimuli_list):

        stim_name = stim['stimulus']

        start = stim['start'] + 0.3
        end = stim['end']

        # obsługa rozszerzenia
        if "." in stim_name:
            image_path = f"{STIMULI_PATH}{stim_name}"
        else:
            image_path = f"{STIMULI_PATH}{stim_name}.jpg"

        if not os.path.exists(image_path):
            print(f"⚠️ Nie znaleziono obrazu: {image_path}")
            continue


        stim_data = df_analysis[
            (df_analysis['t_seconds_tobii'] >= start) &
            (df_analysis['t_seconds_tobii'] <= end)
        ].copy()


        if len(stim_data) < 10:
            print(f"⏩ Pominięto {stim_name} – za mało danych ({len(stim_data)})")
            continue


        outside = stim_data['outside_screen'].mean() * 100

        print(f"🖼️ {stim_name}: {len(stim_data)} próbek, {outside:.1f}% poza ekranem")


        safe_name = "".join(c for c in stim_name if c.isalnum() or c in (' ', '-', '_')).rstrip()
        safe_name = safe_name[:30]

        save_path = f"{output_dir}heatmap_{participant_id}_{safe_name}.png"


        create_face_heatmap(
            data=stim_data,
            image_path=image_path,
            title=f"Bodziec: {stim_name}\nUczestnik: {participant_id}",
            save_path=save_path
        )

In [ ]:
# Komórka 15 - Wywołanie heatmap dla każdego bodźca
if len(stimuli_list) > 0:
    create_stimulus_heatmaps(df_analysis, stimuli_list, OUTPUT_DIR, ID)
else:
    print("⚠️  Brak listy bodźców – nie tworzę heatmap")

In [ ]:
# Komórka 16 - Tworzenie heatmapy dla całej sesji
print("🌐 Tworzenie heatmapy dla całej sesji...")
create_simple_heatmap(
    data=df_analysis,
    title=f"Heatmapa całej sesji\nUczestnik: {ID}",
    save_path=f"{OUTPUT_DIR}heatmap_all_{ID}.png"
)

In [ ]:
# Komórka 17 - Podsumowanie i raport
print("="*60)
print("📊 RAPORT PODSUMOWUJĄCY")
print("="*60)
print(f"👤 Uczestnik: {ID}, Wiek: {AGE}, Płeć: {SEX}")
print(f"📈 Wszystkich wierszy: {len(df_combined)}")
print(f"📈 Wierszy z współrzędnymi: {len(df_analysis)}")
print(f"⏱️  Czas trwania: {df_combined['t_seconds_tobii'].max() - df_combined['t_seconds_tobii'].min():.1f}s")
print(f"👀 % poza ekranem: {df_analysis['outside_screen'].mean()*100:.2f}%")
print(f"🖼️  Liczba bodźców: {len(stimuli_list)}")
print(f"💾 Wyniki zapisane w: {OUTPUT_DIR}")
print("\n✅ ANALIZA ZAKOŃCZONA!")